# UrbanPulse Nepal — Cross-Method Comparison

This notebook brings the measured results from the DBSCAN, hierarchical clustering, and Isolation Forest notebooks into one comparison table. These methods solve different tasks and their metrics are not directly comparable. The goal is to summarize what each method found, how sensitive its output was to configuration, and what evidence is still missing—not to produce one unsupported overall score.

Run notebooks 01–05 first. This notebook reads their saved artifacts and creates a concise model-comparison report for the project documentation.

## 1. Load the results from the same data snapshot

The newest processed incident snapshot is used to locate matching artifacts. The notebook checks that all three method outputs exist before building the comparison.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

PROJECT_ROOT=Path.cwd()
if not (PROJECT_ROOT/'data').is_dir(): PROJECT_ROOT=PROJECT_ROOT.parent
if not (PROJECT_ROOT/'data').is_dir(): raise FileNotFoundError('Run from the repository or notebooks/ folder.')
DATA_ROOT=PROJECT_ROOT/'data'/'processed'/'bipad_incidents'
data_dirs=sorted((p for p in DATA_ROOT.iterdir() if (p/'data_quality_report.json').is_file()),key=lambda p:p.name)
if not data_dirs: raise FileNotFoundError('Run notebooks 01 and 02 first.')
SNAPSHOT_ID=data_dirs[-1].name
data_dir=data_dirs[-1]
dbscan_dir=PROJECT_ROOT/'data'/'processed'/'dbscan'/SNAPSHOT_ID
hier_dir=PROJECT_ROOT/'data'/'processed'/'hierarchical'/SNAPSHOT_ID
if_dir=PROJECT_ROOT/'data'/'processed'/'isolation_forest'/SNAPSHOT_ID
required_files=[dbscan_dir/'dbscan_configuration_comparison.csv',hier_dir/'hierarchical_configuration_comparison.csv',if_dir/'isolation_forest_configuration_comparison.csv']
missing=[str(p) for p in required_files if not p.is_file()]
if missing: raise FileNotFoundError('Run notebooks 03, 04, and 05. Missing artifacts: '+', '.join(missing))
quality=json.loads((data_dir/'data_quality_report.json').read_text(encoding='utf-8'))
dbscan=pd.read_csv(required_files[0]); hierarchical=pd.read_csv(required_files[1]); isolation=pd.read_csv(required_files[2])
print('Snapshot:',SNAPSHOT_ID)
print('Incident rows:',quality.get('rows'))
print('Pagination complete:',quality.get('quality_audit',{}).get('pagination_complete'))
print('DBSCAN configurations:',len(dbscan),'| hierarchical:',len(hierarchical),'| Isolation Forest:',len(isolation))

Snapshot: 20261013T101245Z
Incident rows: 64735
Pagination complete: False
DBSCAN configurations: 5 | hierarchical: 24 | Isolation Forest: 6


## 2. Summarize task, evidence, and limitations

Select representative configurations based on the actual trade-offs in each method. Keep task-specific metrics in their own columns. DBSCAN silhouette is sampled among clustered points; hierarchical silhouette measures the municipality profile partitions; Isolation Forest candidate rate is set by contamination and has no labeled ground truth.

In [2]:
dbscan_high_sil=dbscan.loc[dbscan['sampled_clustered_point_silhouette_haversine'].idxmax()]
dbscan_low_noise=dbscan.loc[dbscan['noise_percent'].idxmin()]
dbscan_tight=dbscan[(dbscan['eps_km']==1.0)&(dbscan['min_samples']==10)].iloc[0]
hier_high=hierarchical.loc[hierarchical['silhouette'].idxmax()]
ward_rows=hierarchical[(hierarchical['linkage']=='ward')&(hierarchical['minimum_incidents']==20)&(hierarchical['n_clusters']==3)]
hier_ward=ward_rows.iloc[0] if len(ward_rows) else hierarchical[hierarchical['linkage']=='ward'].sort_values('cluster_size_ratio').iloc[0]
iforest_1= isolation[(isolation['max_samples']==512)&(isolation['contamination_setting']==0.01)].iloc[0]
iforest_3= isolation[(isolation['max_samples']==512)&(isolation['contamination_setting']==0.03)].iloc[0]

rows=[
 {'method':'DBSCAN','task':'Spatial density clustering of incident locations','configuration':f"eps={dbscan_high_sil['eps_km']:g} km; min_samples={int(dbscan_high_sil['min_samples'])}",'observed_result':f"{int(dbscan_high_sil['clusters_excluding_noise']):,} clusters; {dbscan_high_sil['noise_percent']:.2f}% noise; largest cluster {int(dbscan_high_sil['largest_cluster_size']):,}",'task_specific_evidence':f"Sampled clustered-point Haversine silhouette {dbscan_high_sil['sampled_clustered_point_silhouette_haversine']:.3f}",'limitation':'Many small clusters; score excludes noise and is sample-based. Historical record density is not future risk.'},
 {'method':'DBSCAN','task':'Sensitivity example: broad radius','configuration':f"eps={dbscan_low_noise['eps_km']:g} km; min_samples={int(dbscan_low_noise['min_samples'])}",'observed_result':f"{int(dbscan_low_noise['clusters_excluding_noise']):,} clusters; {dbscan_low_noise['noise_percent']:.2f}% noise; largest cluster {int(dbscan_low_noise['largest_cluster_size']):,}",'task_specific_evidence':f"Sampled clustered-point Haversine silhouette {dbscan_low_noise['sampled_clustered_point_silhouette_haversine']:.3f}",'limitation':'The lowest-noise setting may over-merge distant places into a giant cluster.'},
 {'method':'Hierarchical','task':'Cluster municipalities by hazard and seasonal profile','configuration':f"min incidents={int(hier_high['minimum_incidents'])}; linkage={hier_high['linkage']}; k={int(hier_high['n_clusters'])}",'observed_result':f"{int(hier_high['municipalities'])} municipalities; cluster sizes {int(hier_high['largest_cluster_size'])} largest and {int(hier_high['smallest_cluster_size'])} smallest",'task_specific_evidence':f"Silhouette {hier_high['silhouette']:.3f}",'limitation':'Highest silhouette partition is highly imbalanced; singleton groups make it unsuitable as a final grouping.'},
 {'method':'Hierarchical','task':'More balanced Ward-linkage comparison','configuration':f"min incidents={int(hier_ward['minimum_incidents'])}; Ward linkage; k={int(hier_ward['n_clusters'])}",'observed_result':f"{int(hier_ward['municipalities'])} municipalities; {int(hier_ward['smallest_cluster_size'])}–{int(hier_ward['largest_cluster_size'])} per cluster",'task_specific_evidence':f"Silhouette {hier_ward['silhouette']:.3f}",'limitation':'More balanced but low separation; no robust final municipality segmentation established.'},
 {'method':'Isolation Forest','task':'Rank unusual municipality-month activity profiles','configuration':f"max_samples=512; contamination={iforest_1['contamination_setting']:.0%}",'observed_result':f"{int(iforest_1['candidate_count']):,}/{int(iforest_1['observations']):,} candidates ({iforest_1['observed_candidate_percent']:.2f}%)",'task_specific_evidence':'No ground-truth labels; tested sensitivity across contamination and subsample sizes.','limitation':'Candidates need review; active-month baseline excludes zero months and is not population-adjusted.'},
 {'method':'Isolation Forest','task':'Higher candidate-rate sensitivity setting','configuration':f"max_samples=512; contamination={iforest_3['contamination_setting']:.0%}",'observed_result':f"{int(iforest_3['candidate_count']):,}/{int(iforest_3['observations']):,} candidates ({iforest_3['observed_candidate_percent']:.2f}%)",'task_specific_evidence':'Candidate rate is controlled by contamination; it is not measured anomaly prevalence.','limitation':'No precision/recall or predictive accuracy can be claimed without validated anomaly labels.'}
]
comparison=pd.DataFrame(rows)
display(comparison)

,method,task,configuration,observed_result,task_specific_evidence,limitation
0,DBSCAN,Spatial density clustering of incident locations,eps=1 km; min_samples=10,"1,019 clusters; 32.49% noise; largest cluster ...",Sampled clustered-point Haversine silhouette 0...,Many small clusters; score excludes noise and ...
1,DBSCAN,Sensitivity example: broad radius,eps=5 km; min_samples=5,"298 clusters; 0.99% noise; largest cluster 38,428",Sampled clustered-point Haversine silhouette -...,The lowest-noise setting may over-merge distan...
2,Hierarchical,Cluster municipalities by hazard and seasonal ...,min incidents=10; linkage=average; k=3,726 municipalities; cluster sizes 724 largest ...,Silhouette 0.772,Highest silhouette partition is highly imbalan...
3,Hierarchical,More balanced Ward-linkage comparison,min incidents=20; Ward linkage; k=3,610 municipalities; 10–314 per cluster,Silhouette 0.097,More balanced but low separation; no robust fi...
4,Isolation Forest,Rank unusual municipality-month activity profiles,max_samples=512; contamination=1%,"283/28,202 candidates (1.00%)",No ground-truth labels; tested sensitivity acr...,Candidates need review; active-month baseline ...
5,Isolation Forest,Higher candidate-rate sensitivity setting,max_samples=512; contamination=3%,"847/28,202 candidates (3.00%)",Candidate rate is controlled by contamination;...,No precision/recall or predictive accuracy can...


## 3. Document evidence boundaries and next-use decisions

The methods are complementary, not competitors on a single metric. DBSCAN addresses spatial density, hierarchical clustering addresses municipality profile similarity, and Isolation Forest ranks unusual municipality-month activity. Historical coverage was not verified, and none of these unsupervised outputs is a validated future prediction. A method can be useful for exploration even when the current configuration does not justify a definitive substantive conclusion.

In [3]:
decision_summary={
 'snapshot_id':SNAPSHOT_ID,
 'incident_records':quality.get('rows'),
 'pagination_complete':quality.get('quality_audit',{}).get('pagination_complete'),
 'coverage_note':quality.get('coverage_limitation'),
 'algorithm_roles':{'DBSCAN':'Explore spatial density and noise among recorded incident locations.','Hierarchical clustering':'Compare municipality incident hazard/season profiles; current tested partitions were imbalanced or weakly separated.','Isolation Forest':'Rank unusual municipality-month profiles for manual review; no ground-truth anomaly labels.'},
 'overall_selection':'No single winning model is claimed. Keep the methods as complementary exploratory analyses; review candidate spatial settings and anomaly rows, and do not publish municipality group labels as robust until profile balance and stability improve.'
}
display(pd.Series(decision_summary))
OUTPUT_DIR=PROJECT_ROOT/'reports'
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
table_path=OUTPUT_DIR/'model_comparison.csv'
summary_path=OUTPUT_DIR/'model_comparison_summary.json'
comparison.to_csv(table_path,index=False,encoding='utf-8-sig')
summary_path.write_text(json.dumps(decision_summary,ensure_ascii=False,indent=2),encoding='utf-8')
print('Saved comparison table:',table_path)
print('Saved evidence summary:',summary_path)

snapshot_id                                             20261013T101245Z
incident_records                                                   64735
pagination_complete                                                False
coverage_note          Pagination ended on an empty page, but API cou...
algorithm_roles        {'DBSCAN': 'Explore spatial density and noise ...
overall_selection      No single winning model is claimed. Keep the m...
dtype: object

Saved comparison table: c:\Projects_Tracking\DisasterLens Nepal\reports\model_comparison.csv
Saved evidence summary: c:\Projects_Tracking\DisasterLens Nepal\reports\model_comparison_summary.json


## Findings from the executed cross-method comparison

The comparison used snapshot `20261013T101245Z` and **64,735 incident records**. It compared 5 DBSCAN configurations, 24 hierarchical configurations, and 6 Isolation Forest configurations. The API pagination status is unverified: its reported count was an implausible signed-integer maximum and retrieval ended on an empty page.

| Method and example setting | Measured result | Reading |
|---|---|---|
| DBSCAN, 1 km / 10 samples | 1,019 clusters; 32.49% noise; sampled silhouette 0.779 | Many clusters and a high noise share; the silhouette excludes noise and is sample-based. |
| DBSCAN, 5 km / 5 samples | 298 clusters; 0.99% noise; largest cluster 38,428; sampled silhouette -0.528 | The broad radius merges too many records into a giant cluster in this run. |
| Hierarchical, average linkage, 726 municipalities, 3 clusters | Silhouette 0.772; sizes 724 / 1 / 1 | High score is driven by an unusably imbalanced partition. |
| Hierarchical, Ward, 610 municipalities, 3 clusters | Silhouette 0.097; sizes 314 / 286 / 10 | More balanced than average/complete linkage, but weak separation. |
| Isolation Forest, 512 samples / 1% contamination | 283 candidates out of 28,202 municipality-months (1.00%) | Candidate rate is configured, not validated against labels. |
| Isolation Forest, 512 samples / 3% contamination | 847 / 28,202 candidates (3.00%) | Higher threshold setting returns more candidates for review. |

These metrics answer different questions and must not be ranked against one another. The analysis supports using the methods as complementary historical pattern tools, while withholding any claim of a best predictive model or validated municipality segmentation. DBSCAN clusters are recorded-location density patterns; Isolation Forest outputs require case review; none is a forecast. Incomplete historical coverage, missing zero-incident municipality-months, and unknown reporting exposure remain important limits.